# Лекція 8: MCP — Model Context Protocol: інтеграція AI-інструментів

**Передумови:**
- Лекція 6: HTTP, REST, FastAPI, Pydantic-схеми
- Лекція 7: async/await, httpx, pytest, FastAPI TestClient

**Необхідні інструменти**: Python 3.13+, `pipx`, MCP-сумісний LLM-клієнт (Claude Desktop / Cursor / Gemini CLI)

## Навчальні цілі

Після цієї лекції ви зможете:

1. **Пояснити, що таке MCP** і чому стандартизовані протоколи для AI-інструментів мають значення
2. **Описати архітектуру Host/Client/Server** та три примітиви MCP
3. **Встановити та налаштувати MCP-сервер** (keep-mcp) з автентифікацією
4. **Використовувати MCP-сумісний LLM-клієнт** для взаємодії із зовнішніми сервісами
5. **Застосовувати принципи безпеки** (найменші привілеї, безпечні значення за замовчуванням)
6. **Писати тести**, що мокають виклики MCP-інструментів

---

## 1. Вступ і мотивація

Ми навчились будувати API. Але як **AI-моделі** (Claude, ChatGPT, Copilot) підключаються до зовнішніх сервісів?

Кожна інтеграція — це окремий код, окремі формати, окремі протоколи... Хаос!

**MCP** (Model Context Protocol) вирішує цю проблему.

### 1.1 USB-C для AI

**MCP — це "USB-C" для AI-додатків:**

```
БЕЗ MCP:                               З MCP:
┌───────┐                              ┌───────┐
│Claude │──custom──►Google Keep         │Claude │
│       │──custom──►GitHub              │       │──MCP──►будь-який сервер
│       │──custom──►Slack               │       │
└───────┘                              └───────┘
┌───────┐                              ┌───────┐
│ChatGPT│──custom──►Google Keep         │ChatGPT│
│       │──custom──►GitHub              │       │──MCP──►будь-який сервер
└───────┘                              └───────┘

N хостів × M сервісів = N×M інтеграцій   N хостів + M серверів = N+M
```

> 💡 MCP перетворює проблему **N×M** (кожен AI з кожним сервісом) в проблему **N+M** (стандартний протокол).

### 1.2 MCP у реальному світі

MCP вже використовується у реальних продуктах:
- **Claude Desktop** (Anthropic) — підключається до файлової системи, GitHub, Slack через MCP
- **Cursor** — AI-редактор коду з підтримкою MCP-серверів
- **Windsurf** — ще один AI IDE з MCP
- **Claude Code** — CLI-інструмент Anthropic з MCP-інтеграціями

Для Python-розробників це означає: ви можете писати MCP-сервери, які будь-який AI-клієнт зможе використовувати. Один раз написали — працює скрізь.

> 🎭 *"Пам'ятаєте часи, коли кожен телефон мав свій зарядний кабель? MCP — це момент, коли AI-індустрія нарешті домовилась про один роз'єм."*

![Standardization meme](assets/memes/standards.png)

---

## 2. Архітектура MCP

### 2.1 Три учасники (Host / Client / Server)

![MCP Architecture](https://substackcdn.com/image/fetch/$s_!5Qxi!,w_1200,h_675,c_fill,f_jpg,q_auto:good,fl_progressive:steep,g_auto/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fdc01797d-9996-4b83-a00f-6771b8071d97_900x500.png)

*Джерело: [diamantai](https://diamantai.substack.com/p/model-context-protocol-mcp-explained)*

| Учасник | Роль | Приклади |
|---------|------|----------|
| **Host** (хост) | AI-додаток, що координує клієнтів | Claude Desktop, VS Code, Claude Code |
| **Client** (клієнт) | Компонент, що підтримує з'єднання з одним сервером | Створюється хостом для кожного сервера |
| **Server** (сервер) | Надає контекст через інструменти/ресурси/промпти | keep-mcp, filesystem, DB MCP |

```
┌─────────────────────────────────────┐
│           Host (Claude Desktop)      │
│                                      │
│  ┌──────────┐    ┌──────────┐       │
│  │ Client 1 │    │ Client 2 │       │
│  └────┬─────┘    └────┬─────┘       │
└───────┼───────────────┼─────────────┘
        │               │
   MCP Protocol    MCP Protocol
        │               │
   ┌────▼─────┐    ┌───▼──────┐
   │ Server A │    │ Server B │
   │(keep-mcp)│    │(filesystem)│
   └──────────┘    └──────────┘
```

### 2.2 Три примітиви MCP (MCP Primitives)

MCP-сервер може надавати три типи можливостей:

| Примітив | Хто контролює | Опис | Приклад |
|----------|---------------|------|----------|
| **Tools** (інструменти) | Модель (model) | Функції, які AI може викликати | `create_note()`, `search()` |
| **Resources** (ресурси) | Додаток (app) | Джерела даних для контексту | Вміст файлів, записи БД |
| **Prompts** (промпти) | Користувач (user) | Шаблони для структурованих запитів | "Summarize this document" |

Протокол побудований на **JSON-RPC 2.0** — стандарт виклику функцій через JSON.

> 💡 Зверніть увагу на різницю: **Tools** = AI вирішує коли викликати, **Resources** = додаток вирішує що показати, **Prompts** = користувач вибирає шаблон.

### 2.3 REST-to-MCP: знайомий паттерн

Якщо ви вже знаєте REST — ви вже розумієте MCP tools! Це той самий CRUD, але замість HTTP-ендпоінтів — функції, які AI може викликати.

| REST ендпоінт | HTTP метод | MCP Tool | Тип дії |
|---------------|-----------|----------|----------|
| `/notes/create` | POST | `create_note` | Створення |
| `/notes/search` | POST | `find` | Читання |
| `/notes/{id}` | GET | `get_note` | Читання |
| `/notes/{id}` | DELETE | `trash_note` | Видалення |

In [ ]:
# REST → MCP маппінг: як ендпоінти стають інструментами

rest_endpoints = [
    {"method": "POST", "path": "/notes/create", "description": "Створити нотатку"},
    {"method": "POST", "path": "/notes/search", "description": "Пошук нотаток"},
    {"method": "GET",  "path": "/notes/{id}",   "description": "Отримати нотатку"},
    {"method": "DELETE", "path": "/notes/{id}", "description": "Видалити нотатку"},
]

mcp_tools = [
    {"name": "create_note",  "description": "Створити нотатку"},
    {"name": "search_notes", "description": "Пошук нотаток"},
    {"name": "get_note",     "description": "Отримати нотатку"},
    {"name": "delete_note",  "description": "Видалити нотатку"},
]

print("REST → MCP Mapping:")
print(f"{"REST Endpoint":<30} {"MCP Tool":<20} {"Description"}")
print("-" * 70)
for ep, tool in zip(rest_endpoints, mcp_tools):
    rest = f"{ep["method"]} {ep["path"]}"
    print(f"{rest:<30} {tool["name"]:<20} {tool["description"]}")

### Вправа 1: Визначення примітивів MCP

**Завдання**: Уявіть, що ви створюєте MCP-сервер для бібліотечної системи. Яку функціональність ви б надали через кожен примітив?

Заповніть таблицю:

| Примітив | Що б ви надали? | Приклад |
|----------|----------------|----------|
| Tools | ? | `search_books()`, `borrow_book()` |
| Resources | ? | `books://catalog`, `books://overdue` |
| Prompts | ? | "Порекомендуй книгу по темі..." |

*Подумайте 3-5 хвилин, потім порівняйте з розв'язком нижче.*

### Розв'язок Вправи 1 (натисніть, щоб розгорнути)

<details>
<summary>Показати розв'язок</summary>

| Примітив | Що надаємо | Приклади |
|----------|-----------|----------|
| **Tools** | Дії, які AI може виконувати від імені користувача | `search_books(query)` — пошук книг; `borrow_book(book_id)` — взяти книгу; `return_book(book_id)` — повернути книгу; `reserve_book(book_id)` — зарезервувати |
| **Resources** | Дані, які додаток може показати AI для контексту | `books://catalog` — повний каталог; `books://overdue` — прострочені книги; `books://user/{id}/history` — історія читача |
| **Prompts** | Шаблони запитів, які користувач може вибрати | "Порекомендуй книгу по темі {topic}"; "Згенеруй звіт по простроченим книгам"; "Знайди схожі книги до {title}" |

**Ключова різниця:**
- **Tools** — AI *вирішує* коли їх використовувати (на основі запиту користувача)
- **Resources** — *додаток* вирішує які дані показати AI
- **Prompts** — *користувач* вибирає шаблон і параметри

</details>

---

## 3. Життєвий цикл MCP та транспорти

### 3.1 Як MCP-клієнт запускає сервер? (MCP Lifecycle)

**Важливо**: MCP "сервер" — це **НЕ** традиційний веб-сервер, який слухає порт і чекає з'єднань (як наш FastAPI з `uvicorn`). У стандартному режимі MCP-сервер — це **дочірній процес** (subprocess), який хост **запускає сам**.

Коли ви додаєте конфігурацію в Claude Desktop:

```json
"keep-mcp": {
  "command": "pipx",
  "args": ["run", "keep-mcp"]
}
```

...ви кажете хосту: "Запусти цю **команду** як subprocess і спілкуйся з ним через stdin/stdout."

![MCP Lifecycle](assets/mcp-lifecycle.png)

**Повний життєвий цикл (lifecycle):**

1. **Хост запускається** — наприклад, ви відкриваєте Claude Desktop
2. **Хост читає конфіг** — знаходить запис `keep-mcp` з командою `pipx run keep-mcp`
3. **Хост створює subprocess** — виконує `pipx run keep-mcp` з переданими змінними середовища (`GOOGLE_MASTER_TOKEN`, `GOOGLE_EMAIL`)
4. **Ініціалізація** — хост надсилає `initialize` запит через stdin → сервер відповідає через stdout (обмін можливостями)
5. **Робота** — хост надсилає JSON-RPC запити (`tools/call`, наприклад `find` нотатки) через stdin → сервер відповідає результатом через stdout
6. **Завершення** — хост надсилає `shutdown` → сервер коректно завершується
7. **Процес зникає** — коли хост закривається або користувач від'єднується, subprocess завершується

> 💡 Протокол спілкування — **JSON-RPC 2.0** через stdin/stdout. Це НЕ HTTP-запити. Сервер не слухає жодного порту!

### 3.2 Анотований конфіг + транспорт (stdio vs SSE)

Тепер давайте розберемо кожне поле конфігурації:

```json
{
  "mcpServers": {
    "keep-mcp": {                    // Ім'я сервера (довільне)
      "command": "pipx",             // ЩО запустити (shell-команда)
      "args": ["run", "keep-mcp"],   // ЯК запустити (аргументи команди)
      "env": {                       // Змінні середовища для subprocess
        "GOOGLE_MASTER_TOKEN": "...",
        "GOOGLE_EMAIL": "..."
      }
    }
  }
}
```

Зверніть увагу: тут **немає URL**. Це **shell-команда**, яку хост виконує. Хост стає "батьківським процесом" (parent process), а MCP-сервер — "дочірнім" (child process).

> ⚠️ Ви *можете* запустити `pipx run keep-mcp` вручну в терміналі для налагодження (debugging), але в нормальному режимі хост керує всім автоматично.

**Два типи транспорту (transport) в MCP:**

| Вимір | stdio (Локальний) | HTTP+SSE (Віддалений) |
|-------|:-----------------:|:---------------------:|
| Як сервер запускається | Хост створює subprocess | Сервер працює незалежно (окремий процес/машина) |
| Комунікація | stdin/stdout (JSON-RPC) | HTTP POST (клієнт→сервер) + Server-Sent Events (сервер→клієнт) |
| Мережа потрібна? | Ні (та сама машина) | Так (можуть бути різні машини) |
| Формат конфігу | `"command"` + `"args"` | `"url"` (напр. `"http://mcp.example.com:3000"`) |
| Типовий випадок | Десктоп-інструменти (Claude Desktop, Cursor) | Командні сервери, хмарні розгортання |
| Управління lifecycle | Хост керує (запуск/зупинка) | Сервер керується незалежно (systemd, Docker) |

> 💡 Для локальної роботи (як у нас) завжди використовується **stdio**. SSE/HTTP — для випадків, коли MCP-сервер працює на віддаленому сервері для спільного доступу команди.

Зверніть увагу: SSE (Server-Sent Events) — це просто HTTP-стрімінг, який ви вже знаєте з основ веб-серверів (Лекція 6). Сервер тримає з'єднання відкритим і надсилає дані по мірі їх появи.

In [ ]:
# Приклад: програмне читання MCP-конфігурації
import json

mcp_config = {
    "mcpServers": {
        "keep-mcp": {
            "command": "pipx",
            "args": ["run", "keep-mcp"],
            "env": {
                "GOOGLE_MASTER_TOKEN": "your-token-here",
                "GOOGLE_EMAIL": "your-email@gmail.com"
            }
        }
    }
}

# Виведемо конфіг у форматі JSON
print(json.dumps(mcp_config, indent=2))

# Перевірка: які сервери налаштовані?
for name, cfg in mcp_config["mcpServers"].items():
    print(f"Server: {name}")
    print(f"  Command: {cfg["command"]} {" ".join(cfg["args"])}")
    print(f"  Env vars: {list(cfg.get("env", {}).keys())}")

---

## 4. Практичне налаштування: keep-mcp

### 4.1 Що таке keep-mcp?

[keep-mcp](https://github.com/feuerdev/keep-mcp) — це MCP-сервер, який підключає AI до **Google Keep** (нотатки).

**Інструменти (Tools) keep-mcp** — зверніть увагу на CRUD-маппінг з нашої лекції:

| CRUD | keep-mcp Tool | Що робить |
|------|---------------|----------|
| **C**reate | `create_note`, `create_list` | Створити нотатку/список |
| **R**ead | `find`, `get_note` | Знайти / отримати нотатку |
| **U**pdate | `update_note`, `set_note_color`, `pin_note` | Оновити нотатку |
| **D**elete | `trash_note`, `delete_note` | Видалити нотатку |

**Безпека за замовчуванням** (Safety by Default):
- За замовчуванням keep-mcp працює **тільки** з нотатками, які мають мітку `keep-mcp`
- Щоб працювати з усіма нотатками: `UNSAFE_MODE=true`

### 4.2 Встановлення keep-mcp

```bash
# Крок 1: Встановлення pipx (якщо немає) для MacOS/Linux
pip install pipx
# Для Вінди ідемо сюди: https://scoop.sh/ і встановлюємо scoop
# далі:
scoop install pipx
pipx ensurepath
# Крок 1.1: Встановлення keep-mcp через pipx (ізольоване середовище)
pipx install keep-mcp
```

> 💡 `pipx` встановлює Python-інструменти в ізольовані середовища — не забруднює ваш глобальний Python.

### 4.3 Автентифікація: Google Master Token

keep-mcp використовує бібліотеку [gkeepapi](https://github.com/kiwiz/gkeepapi) для доступу до Google Keep.

**Крок 2**: Отримати master token для авторизації.

Детальна інструкція: [gkeepapi](https://gkeepapi.readthedocs.io/en/latest/#id4)

Як отримати Google Oauth Token: [gpsoauth-java](https://github.com/rukins/gpsoauth-java?tab=readme-ov-file)

```bash
# Після отримання токена — зберегти в змінну середовища:
export GOOGLE_MASTER_TOKEN="your-token-here"
export GOOGLE_EMAIL="your-email@gmail.com"
```

> ⚠️ **Безпека**: Master token дає **повний доступ** до вашого Google-акаунту. Ніколи не комітьте його в git! Використовуйте `.env` або менеджер секретів.

### 4.4 Конфігурація LLM-клієнта

**Крок 3**: Налаштувати ваш LLM-клієнт (host) для підключення до keep-mcp (server).

**Claude Desktop** (основний варіант) — файл `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "keep-mcp": {
      "command": "pipx",
      "args": ["run", "keep-mcp"],
      "env": {
        "GOOGLE_MASTER_TOKEN": "your-token-here",
        "GOOGLE_EMAIL": "your-email@gmail.com"
      }
    }
  }
}
```

Розташування файлу:
- **macOS**: `~/Library/Application Support/Claude/claude_desktop_config.json`
- **Windows**: `%APPDATA%\Claude\claude_desktop_config.json`

**У Вінді може бути проблема з pipx, тому, якщо mcp клієнт не запускається можна прописати у конфігурації прямий шлях:**
```json
{
  "mcpServers": {
    "keep-mcp": {
      "command": "C:\\Users\\bohda\\.local\\bin\\mcp.exe",
      "args": [],
      "env": {
        "GOOGLE_MASTER_TOKEN": "your-token",
        "GOOGLE_EMAIL": "your-email"
      }
    }
  }
}
```

**Cursor** (альтернатива):
```json
{
  "mcpServers": {
    "keep-mcp": {
      "command": "pipx",
      "args": ["run", "keep-mcp"]
    }
  }
}
```

**Gemini CLI** (альтернатива) — файл `~/.gemini/settings.json`:
```json
{
  "mcpServers": {
    "keep-mcp": {
      "command": "pipx",
      "args": ["run", "keep-mcp"]
    }
  }
}
```

Після цієї конфігурації Claude зможе створювати, шукати та оновлювати ваші нотатки в Google Keep!

### 4.5 Перевірка з'єднання

**Крок 4**: Після конфігурації перезапустіть LLM-клієнт і перевірте:

1. У Claude Desktop з'явиться іконка 🔌 з переліком MCP-серверів
2. Попросіть Claude: *"Покажи мої нотатки в Google Keep з міткою keep-mcp"*
3. Claude повинен запитати дозвіл на використання інструменту `find` або `get_note`

### 4.6 Усунення проблем (Troubleshooting)

| Проблема | Рішення |
|----------|--------|
| `pipx: command not found` | `pip install --user pipx` та додати до PATH |
| `keep-mcp not found` | `pipx install keep-mcp` (не `pip install`) |
| Token помилка | Перевірити `GOOGLE_MASTER_TOKEN` та `GOOGLE_EMAIL` |
| Claude не бачить MCP | Перезапустити Claude Desktop після зміни конфігу |
| `Connection refused` | Перевірити, що `pipx run keep-mcp` працює в терміналі |
| Нотатки не знаходяться | За замовчуванням keep-mcp бачить лише нотатки з міткою `keep-mcp` |

### Вправа 2: Встановлення та використання keep-mcp

**Завдання**: Виконайте наступні кроки:

1. Встановіть keep-mcp: `pipx install keep-mcp`
2. Налаштуйте автентифікацію (Google Master Token)
3. Налаштуйте ваш LLM-клієнт (Claude Desktop або інший)
4. Виконайте 3 операції через LLM:
   - Знайдіть нотатки з міткою `keep-mcp`
   - Створіть нову нотатку
   - Перелічіть всі нотатки

---

## 5. Безпека

### 5.1 Safe Mode та UNSAFE_MODE

keep-mcp демонструє важливий принцип — **безпечні значення за замовчуванням** (safe defaults):

- **Safe mode** (за замовчуванням): працює тільки з нотатками, що мають мітку `keep-mcp`
- **`UNSAFE_MODE=true`**: дає доступ до ВСІХ нотаток — використовуйте лише коли розумієте ризики

### 5.2 Принцип найменших привілеїв (Principle of Least Privilege)

Давайте інструменту **лише ті дозволи, які йому потрібні**. Не більше.

- Master token дає повний доступ → це **порушення** принципу
- Safe mode обмежує scope → це **дотримання** принципу

### 5.3 Гігієна облікових даних (Credential Hygiene)

- Зберігайте токени у `.env` файлі
- Додайте `.env` до `.gitignore`
- Ніколи не комітьте секрети в git
- Використовуйте менеджери секретів для продакшн-середовищ

### 5.4 Реальні ризики AI-інструментів

- **Prompt injection** — зловмисний текст у нотатках може змусити AI виконати небажані дії
- **Ненавмисні дії** — AI може видалити або змінити дані, які ви не планували
- **Витік даних** — AI може відправити конфіденційні дані на зовнішні сервери

### 5.5 Анотації інструментів (Tool Annotations)

MCP дозволяє серверам анотувати інструменти для додаткової безпеки:

| Анотація | Опис | Приклад |
|----------|------|----------|
| `readOnlyHint` | Інструмент лише читає дані | `find`, `get_note` |
| `destructiveHint` | Інструмент може видалити/змінити дані | `delete_note`, `trash_note` |
| `openWorldHint` | Інструмент взаємодіє із зовнішніми сервісами | `create_note` (створює в Google) |

Ці анотації допомагають LLM-клієнту приймати рішення: наприклад, запитувати підтвердження перед деструктивними діями.

> 🎭 *"AI з доступом до delete_note без підтвердження — це як дати стажеру root-доступ до продакшн-бази в п'ятницю ввечері."*

In [ ]:
# Приклад: анотації інструментів (Tool Annotations)
# Так виглядає опис MCP-інструменту з анотаціями безпеки

tool_definitions = [
    {
        "name": "find",
        "description": "Search notes by query",
        "annotations": {"readOnlyHint": True, "destructiveHint": False},
    },
    {
        "name": "create_note",
        "description": "Create a new note",
        "annotations": {"readOnlyHint": False, "openWorldHint": True},
    },
    {
        "name": "delete_note",
        "description": "Delete a note permanently",
        "annotations": {"readOnlyHint": False, "destructiveHint": True},
    },
]

for tool in tool_definitions:
    safety = []
    ann = tool["annotations"]
    if ann.get("readOnlyHint"): safety.append("✅ read-only")
    if ann.get("destructiveHint"): safety.append("⚠️ DESTRUCTIVE")
    if ann.get("openWorldHint"): safety.append("🌐 external")
    print(f"{tool["name"]:<15} {" | ".join(safety)}")

---

## 6. Тестування MCP-інтеграцій

### Чому мокати MCP-виклики?

MCP-сервери залежать від зовнішніх сервісів (Google Keep, GitHub, тощо). У тестах ми **не хочемо**:
- Залежати від мережі
- Змінювати реальні дані
- Потребувати автентифікацію

Рішення: **мокаємо** (mock) виклики MCP-інструментів за допомогою `monkeypatch` з pytest.

In [ ]:
# Тестування функції, що використовує MCP-інструмент
import pytest


# Уявімо функцію, що викликає MCP-сервер
def get_notes_from_keep(query: str) -> list[dict]:
    """Виклик keep-mcp для пошуку нотаток."""
    # В реальності тут виклик MCP-сервера
    ...


def test_get_notes_mocked(monkeypatch):
    """Мокаємо MCP-виклик для тестування без реального сервера."""
    fake_notes = [
        {"title": "Test Note", "content": "Hello MCP", "labels": ["keep-mcp"]}
    ]

    monkeypatch.setattr(
        "app.services.mcp_client.get_notes_from_keep",
        lambda query: fake_notes,
    )

    result = get_notes_from_keep("test")
    assert len(result) == 1
    assert result[0]["title"] == "Test Note"

In [ ]:
import os
import pytest

SKIP_MCP = not os.getenv("MCP_INTEGRATION_TESTS")


@pytest.mark.skipif(SKIP_MCP, reason="MCP integration tests disabled")
def test_keep_mcp_real_connection():
    """Запускається лише при MCP_INTEGRATION_TESTS=1."""
    # Реальний виклик до keep-mcp
    ...

---

## 7. Зв'язок з нашим проєктом

Наш notes-api вже має CRUD-ендпоінти. Уявіть: що, якщо ми зробимо його MCP-сервером?

| notes-api ендпоінт | MCP Tool | Опис |
|-------------------|----------|------|
| POST /notes/create | `create_note` | Створити нотатку |
| POST /notes/search | `search_notes` | Пошук нотаток |
| GET /notes/{id} | `get_note` | Отримати нотатку |
| DELETE /notes/{id} | `delete_note` | Видалити нотатку |

**Що потрібно додати:**

1. **MCP-сервер обгортка** — код, що реєструє tools і обробляє JSON-RPC запити
2. **Tool definitions** — опис кожного інструменту (ім'я, опис, параметри)
3. **Transport setup** — stdio або SSE залежно від потреб

Це концептуальний превʼю — ми **НЕ** реалізуємо це зараз.

---

**Домашнє завдання (необов'язкове):** Намалюйте схему MCP-сервера для notes-api. Які tools, resources та prompts ви б визначили?

---

## Підсумок

1. **MCP = "USB-C для AI"** — стандартний протокол підключення AI до зовнішніх інструментів. Замість N×M кастомних інтеграцій — N+M через єдиний стандарт.
2. **Архітектура: Host → Client → Server** — хост (Claude Desktop) координує клієнтів, кожен клієнт підключається до одного MCP-сервера.
3. **Три примітиви** — Tools (модель вирішує), Resources (додаток вирішує), Prompts (користувач вирішує).
4. **Життєвий цикл** — хост запускає MCP-сервер як subprocess → stdio комунікація через JSON-RPC 2.0 → хост завершує сервер.
5. **Практика: keep-mcp** — `pipx install keep-mcp` → конфігурація LLM-клієнта → автентифікація → перевірка з'єднання.
6. **Безпека** — safe mode за замовчуванням, `UNSAFE_MODE=true` лише коли розумієте ризики, принцип найменших привілеїв, анотації інструментів (`readOnlyHint`, `destructiveHint`).
7. **Тестування** — мокайте MCP-виклики через `monkeypatch`, використовуйте `@pytest.mark.skipif` для інтеграційних тестів.

---

## Що далі?

### Лекція 9: Docker + PostgreSQL — контейнеризація нашого API та підключення реальної бази даних

У наступній лекції ми:
- Вивчимо **Docker** — контейнеризацію додатків для послідовного розгортання
- Підключимо **PostgreSQL** — реальну реляційну базу даних замість in-memory словника
- Налаштуємо **Docker Compose** для запуску API + бази даних одною командою
- Перенесемо наш notes-api з тимчасового сховища на постійне збереження даних

---

## Джерела (References)

### MCP
- [Introducing the Model Context Protocol](https://www.anthropic.com/news/model-context-protocol) — Анонс MCP від Anthropic
- [MCP — Official Docs](https://modelcontextprotocol.io/) — Офіційна документація MCP
- [MCP — Architecture](https://modelcontextprotocol.io/docs/learn/architecture) — Архітектура протоколу
- [Build an MCP server](https://modelcontextprotocol.io/docs/develop/build-server) — Як побудувати свій MCP сервер

### Інструменти
- [keep-mcp](https://github.com/feuerdev/keep-mcp) — MCP-сервер для Google Keep
- [gkeepapi](https://github.com/kiwiz/gkeepapi) — Python-бібліотека для Google Keep